# Excepion, pickle expression régulière

## Exception

In [1]:
1/0

ZeroDivisionError: division by zero

In [2]:
def f(a, b):
    return a + b

f(5)

TypeError: f() missing 1 required positional argument: 'b'

In [3]:
3 + "4"

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [4]:
if True:
print(4)

IndentationError: expected an indented block after 'if' statement on line 1 (1787780864.py, line 2)

In [9]:
try:
    1/0
except TypeError as e1:
    print("erreur 1", e1)
except ZeroDivisionError as e2:
    print("erreur 2", e2)

erreur 2 division by zero


In [12]:
def division(a, b):
    try:
        return a / b
    except ZeroDivisionError as e:
        raise ValueError(f"une erreur a eu lieu avec les nombres {a} et {b}")
division(1, 0)

ValueError: une erreur a eu lieu avec les nombres 1 et 0

In [22]:
def division(a, b):
    try:
        return a / b
    except Exception as e:
        # raise ValueError(f"une erreur a eu lieu avec les nombres {a!r} et {b!r}") from e
        pass
    finally:
        print(f"FIN {a!r} et {b!r}")
    print("il y a une erreur")
    return 0

def divisions(list_a, list_b):
    return sum( [ division(a,b) for a, b in zip(list_a, list_b)])

print("*", divisions([1, 2], [1, 2]))

FIN 1 et 1
FIN 2 et 2
* 2.0


In [24]:
class MonException(RuntimeError):
    pass

def division(a, b):
    try:
        return a / b
    except Exception as e:
        raise MonException(f"calcul impossible {a!r}/{b!r}") from e
    finally:
        print(f"FIN {a!r} et {b!r}")
    print("il y a une erreur")
    return 0

def divisions(list_a, list_b):
    return sum( [ division(a,b) for a, b in zip(list_a, list_b)])

print("*", divisions([1, 2], [1, "2"]))

FIN 1 et 1
FIN 2 et '2'


MonException: calcul impossible 2/'2'

In [34]:
class Toto:
    def __init__(self, age, prenom, nom):
        self.age = age
        self.prenom = prenom
        self.nom = nom
    def est_mineur(self):
        return self.age < 18
    def __repr__(self):
        return f"Toto({self.age}, {self.prenom!r}, {self.nom!r})"
    def rienavoir(self):
        print("je suis Toto.rienavoir")
        return self.age * 10
    def rienavoir_qui_utilise_rienavoir(self):
        print("je suis Toto.rienavoir_qui_utilise_rienavoir")
        return self.rienavoir() + len(self.nom)

t = Toto(16, "titi", "titifamily")
print("t=", t, t.est_mineur(), t.rienavoir_qui_utilise_rienavoir())


je suis Toto.rienavoir_qui_utilise_rienavoir
je suis Toto.rienavoir
t= Toto(16, 'titi', 'titifamily') True 170


In [35]:
class Toto2(Toto):
    def __repr__(self):
        return f"Toto2({self.age}, {self.prenom!r}, {self.nom!r})"
    def rienavoir(self):
        print("je suis Toto2.rienavoir")
        return self.age * -10

t2 = Toto2(16, "titi", "titifamily")
print("t2=", t2, t2.est_mineur(), t2.rienavoir_qui_utilise_rienavoir())

je suis Toto.rienavoir_qui_utilise_rienavoir
je suis Toto2.rienavoir
t2= Toto2(16, 'titi', 'titifamily') True -150


## scikit-learn

In [36]:
import pandas as pd
import numpy as np

data = {
    "age": [25, 32, 47, np.nan, 51],
    "revenu": [50000, 64000, 120000, 75000, np.nan],
    "ville": ["Paris", "Lyon", "Marseille", "Paris", "Lyon"],
    "sexe": ["F", "M", "M", "F", np.nan],
    "achat": [0, 1, 1, 0, 1],
}

df = pd.DataFrame(data)
df

,age,revenu,ville,sexe,achat
0,25.0,50000.0,Paris,F,0
1,32.0,64000.0,Lyon,M,1
2,47.0,120000.0,Marseille,M,1
3,NaN,75000.0,Paris,F,0
4,51.0,NaN,Lyon,NaN,1


In [39]:
def function_categorie(df, column, fill_nan):
    unique = df[column].fillna(fill_nan).unique()
    #d = {}
    #for c in unique:
    #    d[c] = len(d)
    d = {"M": 4, "F": 6, "I": 10}
    df[f"{column}_num"] = df[column].fillna(fill_nan).apply(lambda v: d[v])
    return df.drop("sexe", axis=1)

function_categorie(df, "sexe", "I")

,age,revenu,ville,achat,sexe_num
0,25.0,50000.0,Paris,0,6
1,32.0,64000.0,Lyon,1,4
2,47.0,120000.0,Marseille,1,4
3,NaN,75000.0,Paris,0,6
4,51.0,NaN,Lyon,1,10


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

pipe = Pipeline(
    steps=[
        ("preprocessing", ColumnTransformer(
                transformers=[
                    ("ville_ord", OrdinalEncoder(), ["ville"]),
                    ("sexe_ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["sexe"]),
                ],
                remainder="drop"
        )),
        ("pca", PCA(n_components=3)),
    ]
)

# Entraînement
pipe.fit(df[["sexe", "ville"]])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('pca', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ville_ord', ...), ('sexe_ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [62]:
from sklearn.base import BaseEstimator, TransformerMixin

def function_categorie(df, column, fill_nan):
    unique = df[column].fillna(fill_nan).unique()
    #d = {}
    #for c in unique:
    #    d[c] = len(d)
    d = {"M": 4, "F": 6, "I": 10}
    df[f"{column}_num"] = df[column].fillna(fill_nan).apply(lambda v: d[v])
    return df.drop("sexe", axis=1)
    
class MonEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        assert X.shape[1] == 1, f"X.shape={X.shape}: soit c'est vide, soit il y a plus d'une colonne"
        valeurs = set(X.iloc[:, 0].fillna("I"))
        assert valeurs <= {"M", "F", "I"}, f"Valeurs inattendues {valeurs}"
        return self
        
    def transform(self, X):
        return function_categorie(X, X.columns[0], "I")
    

In [63]:
pipe = Pipeline(
    steps=[
        ("preprocessing", ColumnTransformer(
                transformers=[
                    ("ville_ord", OrdinalEncoder(), ["ville"]),
                    ("sexe_ohe", MonEncoder(), ["sexe"]),
                ],
                remainder="drop"
        )),
    ]
)

# Entraînement
pipe.fit(df[["sexe", "ville"]])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ville_ord', ...), ('sexe_ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [57]:
pipe.transform(df[["sexe", "ville"]])

array([[ 2.,  6.],
       [ 0.,  4.],
       [ 1.,  4.],
       [ 2.,  6.],
       [ 0., 10.]])

In [67]:
def function_categorie(df, column, fill_nan, mapping):
    unique = df[column].fillna(fill_nan).unique()
    d = mapping
    df[f"{column}_num"] = df[column].fillna(fill_nan).apply(lambda v: d[v])
    return df.drop("sexe", axis=1)
    
class MonEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, mapping):
        self.mapping = mapping
    def fit(self, X, y=None):
        assert X.shape[1] == 1, f"X.shape={X.shape}: soit c'est vide, soit il y a plus d'une colonne"
        valeurs = set(X.iloc[:, 0].fillna("I"))
        assert valeurs <= {"M", "F", "I"}, f"Valeurs inattendues {valeurs}"
        return self
        
    def transform(self, X):
        return function_categorie(X, X.columns[0], "I", self.mapping)

pipe = Pipeline(
    steps=[
        ("preprocessing", ColumnTransformer(
                transformers=[
                    ("ville_ord", OrdinalEncoder(), ["ville"]),
                    ("sexe_ohe", MonEncoder(mapping={"M": 50, "F": 55, "I": 160}), ["sexe"]),
                ],
                remainder="drop"
        )),
    ]
)

# Entraînement
pipe.fit(df[["sexe", "ville"]])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ville_ord', ...), ('sexe_ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [66]:
pipe.transform(df[["sexe", "ville"]])

array([[  2.,  55.],
       [  0.,  50.],
       [  1.,  50.],
       [  2.,  55.],
       [  0., 160.]])

## Expressions régulières

In [81]:
phrase = "J'ai payé 9.000 euros pour un tableau."
# 9.000; 9000; 9,000; 9000.00
# r"\b\d+(?:[.,]\d{3})*(?:\.\d{2})?\b"
# \b frontière
# \d+ un chiffre ou plus  [0123456789] ou [0-9]
# \d{3} 3 chiffres
# [.,]  un . ou ,
# [.,]\d{3} = un . ou , suivi de 3 chiffres
# (?:[.,]\d{3})*  = (un . ou , suivi de 3 chiffres) répété 0 ou +
# (?:\.\d{2})? = (. suivi de 2 chiffres) présent ou pas
import re
reg = re.compile(r"\b(\d+([.,]\d{3})*(\.\d{2})?)\b")
match = reg.search(phrase)
match

<re.Match object; span=(10, 15), match='9.000'>

In [82]:
match.groups()

('9.000', '.000', None)

In [80]:
match.group(0)

'9.000'

In [83]:
phrase = "J'ai payé 9.000 euros pour un tableau, ou 1.000.00, je ne sais plus."

In [84]:
match = reg.search(phrase, pos=0)
match

<re.Match object; span=(10, 15), match='9.000'>

In [85]:
reg.search(phrase, pos=15)

<re.Match object; span=(42, 50), match='1.000.00'>

In [86]:
reg.search(phrase, pos=50)

In [87]:
reg.findall(phrase)

[('9.000', '.000', ''), ('1.000.00', '.000', '.00')]